<a href="https://colab.research.google.com/github/sandrapoer/MCMLR_2024W/blob/main/Bonus_Exercise_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bonus Exercises 2: Low-Rank Adaptation and Crosslingual Transfer**



This notebook represents the second bonus exercises for the lecture Multilingual and Crosslingual Methods and Language Resources (2024W 340168-1). For each successfully completed bonus exercise, a maximum of three points can be achieved that will be added to the points of the final exam. The tasks to be completed in the following notebook are marked with 👋 ⚒.

---




-----------
## **Fine-Tuning on English**

The first part has already been prepared for you. We will load and preprocess the Corpus for Linguistic Acceptability (COLA) dataset from GLUE and then use Low-Rank Adaptation to fine-tune XLM-R.

### Installation

As always, we first need to install the necessary libraries. One that is new in this notebook is the Parameter-Efficient Fine-Tuning (PEFT) library.

In [1]:
!pip install -U evaluate
!pip install -U datasets
!pip install -U transformers
!pip install -U peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 113.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


### Loading the Dataset

In this notebook we will first be using the COLA dataset from the GLUE library and then a multilingual extension.
 We will first train on English and transfer to another language and evaluate zero-shot transfer on one more language (see [here](https://huggingface.co/datasets/Geralt-Targaryen/MELA) for a selection).

In [2]:
from datasets import load_dataset

dataset_en = load_dataset("glue", "cola")
dataset_en.num_rows

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1063 [00:00<?, ? examples/s]

{'train': 8551, 'validation': 1043, 'test': 1063}

Let us take a look at the components of the dataset.

In [3]:
dataset_en['train'].features

{'sentence': Value('string'),
 'label': ClassLabel(names=['unacceptable', 'acceptable']),
 'idx': Value('int32')}

Hugging Face Datasets is designed to be interoperable with libraries like Pandas, as well as NumPy, PyTorch, TensorFlow, and JAX. To enable the conversion between various third-party libraries, Hugging Face Datasets provides a Dataset.set_format() function. This function only changes the output format of the dataset, so you can easily switch to another format without affecting the underlying data format which is Apache Arrow. The formatting is done in-place, so let’s convert our dataset to Pandas and look at a random sample:

In [4]:
from IPython.display import display, HTML

dataset_en.set_format("pandas")
df = dataset_en["train"][:]
# Create a random sample
sample = df.sample(n=5, random_state=42)
display(HTML(sample.to_html()))

,sentence,label,idx
2389,Angela characterized Shelly as a lifesaver.,1,2389
5048,They're not finding it a stress being in the same office.,1,5048
3133,Paul exhaled on Mary.,0,3133
5955,I ordered if John drink his beer.,0,5955
625,Press the stamp against the pad completely.,1,625


The Pandas dataframe can now be used as we would always use Pandas, for instance to count the number of labels for `cause` in the column question.

In [5]:
df["label"].value_counts()

,count
label,
1,6023
0,2528


We can see that the two labels are spread quite evenly across the two types of questions.

This was just a brief detour to show how datasets can be nicely manipulated and displayed using other libraries. We will now get back to our usual datasets library from Hugging Face. To this end, we will reset the format.

In [6]:
dataset_en.reset_format()

### Preprocessing the Dataset

In this example, we model COPA as a multiple-choice task with two choices. Thus, we encode the premise and question as well as both choices as one input to our `xlm-roberta-base` model. Using `dataset.map()`, we can pass the full dataset through the tokenizer in batches.

In [7]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")
batch_size = 32

def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding=True, truncation=True)

def preprocess_dataset(dataset):
  token_dataset = dataset.map(tokenize_function, batched=True, batch_size=batch_size)
  tokenized_dataset = token_dataset.rename_column("label", "labels")
  return tokenized_dataset

data_set_en_with_test = DatasetDict(
    train=dataset_en['train'].shuffle(seed=24).select(range(7488)),
    validation=dataset_en['validation'],
    test=dataset_en['train'].shuffle(seed=24).select(range(7488, 8551)),
)

tokenized_dataset_en = preprocess_dataset(data_set_en_with_test)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/7488 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

In [8]:
tokenized_dataset_en["train"][1]

{'sentence': 'John knows that she left and whether she will come back.',
 'labels': 1,
 'idx': 7246,
 'input_ids': [0,
  4939,
  93002,
  450,
  2412,
  25737,
  136,
  36766,
  2412,
  1221,
  1380,
  4420,
  5,
  2,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

-----------
## **Low-Rank Adaptation (LoRA)**

In order to perform low-rank adaptation (LoRA) on a pretrained language model for parameter-efficient fine-tuning (PEFT), we need to set a few parameters in the LoRA Configuration. Hugging Face offers some [documentation on LoRA](https://huggingface.co/docs/peft/main/en/developer_guides/lora).

The `task-type` specifies which task the model should be fine-tuned on and needs to correspond to the way the model is loaded. If we load a model for Sequence Classification, also the task needs to be `SEQ_CLS`, an abbreviation for Sequence Classification. Then the dataset needs to be one with an input sequence and a number of target classes.

The `target-module` depends on the type of model, which for XLM-R is `["query", "value"]`. Since we wish to change model parameters, the inference mode is set to false. The variable `r`indicates the rank to which the dimensionality is being reduced. The variable `alpha` is a scaling parameter, because `r`scales at 1.0. With small datasets or if unsure, the rank and alpha can be the same. Finally, dropout is a random omission of trainable parameters (setting to zero) during training, mostly to avoid overfitting.

Feel free to play with and adapt these parameters if you are interested in seeing the effect.


In [10]:
!pip install -q -U torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.1 MB/s eta 0:00:00


In [11]:
from peft import LoraConfig, PeftType, get_peft_model
from transformers import AutoModelForSequenceClassification

peft_type = PeftType.LORA
peft_config = LoraConfig(task_type="SEQ_CLS", target_modules=["query", "value"], inference_mode=False, r=8, lora_alpha=8, lora_dropout=0.1)
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-large")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,838,082 || all params: 561,730,564 || trainable%: 0.3272


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): XLMRobertaForSequenceClassification(
      (classifier): ModulesToSaveWrapper(
        (original_module): XLMRobertaClassificationHead(
          (dense): Linear(in_features=1024, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (out_proj): Linear(in_features=1024, out_features=2, bias=True)
        )
        (modules_to_save): ModuleDict(
          (default): XLMRobertaClassificationHead(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (out_proj): Linear(in_features=1024, out_features=2, bias=True)
          )
        )
      )
      (roberta): XLMRobertaModel(
        (embeddings): XLMRobertaEmbeddings(
          (word_embeddings): Embedding(250002, 1024, padding_idx=1)
          (token_type_embeddings): Embedding(1, 1024)
          (LayerNorm): LayerNorm((1024,), eps=1e-05, e

In [13]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer, EvalPrediction
from datasets import concatenate_datasets

num_train_epochs = 5
logging_steps = len(tokenized_dataset_en["train"]) // (batch_size * num_train_epochs)
accuracy = evaluate.load("accuracy")

training_args = TrainingArguments(
    learning_rate=2e-4,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=logging_steps,
    output_dir="./training_output",
    report_to='none',
    load_best_model_at_end=True,
    # The next line is important to ensure the dataset labels are properly passed to the model
    remove_unused_columns=True,
)

def compute_metrics(eval_pred):
    """Called at the end of validation. Gives accuracy"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # calculates the accuracy
    return accuracy.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_en["train"],
    eval_dataset=tokenized_dataset_en["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Once we have configured the model with PEFT, we can train the PEFT model as usual.

In [14]:
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.601961,0.596106,0.691275
2,0.553871,0.517954,0.751678
3,0.495192,0.506072,0.782359
4,0.462444,0.543095,0.777565
5,0.418383,0.526823,0.780441


TrainOutput(global_step=1170, training_loss=0.5241119458125187, metrics={'train_runtime': 599.9467, 'train_samples_per_second': 62.406, 'train_steps_per_second': 1.95, 'total_flos': 2688914965511424.0, 'train_loss': 0.5241119458125187, 'epoch': 5.0})

👋 ⚒ Evaluate on the English test set to see how well the fine-tuning has worked.

In [15]:
# Your code for the evaluation here
import torch
from peft import PeftModel, PeftConfig

peft_model_id = "/content/training_output/checkpoint-702"
config = PeftConfig.from_pretrained(peft_model_id)
inference_model = AutoModelForSequenceClassification.from_pretrained(config.base_model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

inference_model = PeftModel.from_pretrained(inference_model, peft_model_id)

model_inputs = tokenizer("I ordered if John dink his beer.", return_tensors="pt")
outputs = inference_model(**model_inputs)
prediction = outputs.logits.argmax(dim=-1)
print(prediction)
print(["unacceptable", "acceptable"][prediction])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([0])
unacceptable


## **Crosslingual Transfer**

In this section, we will be using the Multilingual Evaluation of Linguistic Acceptability ([MELA](https://github.com/sjtu-compling/mela?tab=readme-ov-file)), which is also [available on Hugging Face](https://huggingface.co/datasets/Geralt-Targaryen/MELA) to test the transfer and zero-shot capabilities of XLM-R with LoRA Fine-Tuning.

We will first fine-tune on German and then test the on German but also in a zero-shot approach on another language of your choice.

Please be aware of the fact that MELA "only" offers a dev and a test set - no train, validation, test split. Thus, the preprocessing needs to be slightly adapted.

In [16]:
from datasets import load_dataset

de = load_dataset("Geralt-Targaryen/MELA", "de")
dataset_de = preprocess_dataset(de)
print(dataset_de["test"][20])

README.md: 0.00B [00:00, ?B/s]

dev.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating dev split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/945 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/945 [00:00<?, ? examples/s]

{'idx': 'c1-1.1_n9-a', 'labels': 1, 'sentence': 'Wenn du glaubst, dass er sich geirrt habe, kannst du dann alles verstehen', 'input_ids': [0, 7896, 115, 24682, 5829, 271, 4, 1421, 72, 833, 6, 128696, 3198, 3260, 4, 32540, 115, 3700, 4174, 85516, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


👋 ⚒ Use the German dev partition to further-finetune the previously configured model and then evaluate on the test partition of the German dataset.

In [18]:
# Fine-tuning on German dev set here
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np

accuracy_metric = evaluate.load("accuracy")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args_de = TrainingArguments(
    learning_rate=2e-4,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    output_dir="./training_output_de",
    report_to="none",
    load_best_model_at_end=True,
    remove_unused_columns=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

trainer_de = Trainer(
    model=model,
    args=training_args_de,
    train_dataset=dataset_de["dev"],
    eval_dataset=dataset_de["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Fine-tuning on German dev set...")
trainer_de.train()

print("Evaluating on German test set...")
results_de = trainer_de.evaluate(dataset_de["test"])
print(f"German test set accuracy: {results_de['eval_accuracy']:.4f} ({results_de['eval_accuracy']*100:.2f}%)")


Fine-tuning on German dev set...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.499037,0.734392
2,No log,0.537892,0.704762
3,No log,0.518732,0.717460


Evaluating on German test set...


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.499037,3,0.734392


German test set accuracy: 0.7344 (73.44%)


👋 ⚒ Select another language of your choice from the [MELA dataset](https://huggingface.co/datasets/Geralt-Targaryen/MELA) to only evaluate the fine-tuned model (zero-shot capability).

**Alternative**: Feel free to create your own mini-dataset of a few (non)-acceptable sentences in a language of your choice to test the model's zero-shot capacity.

In [19]:
# Your evaluation here
fr_mela = load_dataset("Geralt-Targaryen/MELA", "fr")
dataset_fr = preprocess_dataset(fr_mela)
print(f"French MELA test set: {len(dataset_fr['test'])} examples")
print(f"French MELA dev set:  {len(dataset_fr['dev'])} examples")

# Zero-shot
accuracy_metric = evaluate.load("accuracy")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.to(device)

all_predictions = []
all_labels = []

for example in dataset_fr["test"]:
    input_ids = torch.tensor([example["input_ids"]]).to(device)
    attention_mask = torch.tensor([example["attention_mask"]]).to(device)
    label = example["labels"]

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    prediction = outputs.logits.argmax(dim=-1).item()
    all_predictions.append(prediction)
    all_labels.append(label)

results_fr = accuracy_metric.compute(predictions=all_predictions, references=all_labels)
print(f"French zero-shot accuracy: {results_fr['accuracy']:.4f} ({results_fr['accuracy']*100:.2f}%)")
print(f"Total examples evaluated:  {len(all_labels)}")
print()

# self-written examples sentences
demo_sentences_fr = [
    ("Le chat est assis sur le tapis.", "acceptable"),
    ("Je mangé hier le pain.", "unacceptable"),
    ("Elle va à l'école tous les jours.", "acceptable"),
    ("Les oiseaux voler dans le ciel.", "unacceptable"),
]
print("Demo predictions on French sentences:")
for sentence, expected in demo_sentences_fr:
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = ["unacceptable", "acceptable"][outputs.logits.argmax(dim=-1).item()]
    match = "OK" if pred == expected else "WRONG"
    print(f"  [{match}] '{sentence}' -> {pred} (expected: {expected})")


dev.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating dev split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1333 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/1333 [00:00<?, ? examples/s]

French MELA test set: 1333 examples
French MELA dev set:  100 examples
French zero-shot accuracy: 0.8260 (82.60%)
Total examples evaluated:  1333

Demo predictions on French sentences:
  [OK] 'Le chat est assis sur le tapis.' -> acceptable (expected: acceptable)
  [OK] 'Je mangé hier le pain.' -> unacceptable (expected: unacceptable)
  [OK] 'Elle va à l'école tous les jours.' -> acceptable (expected: acceptable)
  [WRONG] 'Les oiseaux voler dans le ciel.' -> acceptable (expected: unacceptable)


In [20]:
# Clean up widget metadata before saving
import IPython
ipy = IPython.get_ipython()
if hasattr(ipy, 'kernel') and hasattr(ipy.kernel, 'comm_manager'):
    ipy.kernel.comm_manager.comms.clear()